# Fixed-target RMU: algorithm and training

Run the cells in order. The CPU example calls the same objective implementation as the production trainer. The full training cell runs the actual algorithm separately on all six pinned study models, with reviewed streams and staged checkpoints. Results are optional.

The fixed seeded target is shared across batches. Production training captures the selected decoder layer, applies the reviewed completion masks, preserves reference retain activations and updates only the configured projections.

## Setup

Start with [setup.ipynb](setup.ipynb) or the [setup guide](SETUP.md). Install `requirements-unlearning.txt` in an isolated environment. The setup configures all six pinned checkpoints and the reviewed data directories. The CPU algorithm example needs only PyTorch.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

repo = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "src/fairforget/unlearning/objectives.py").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError(
        "Open this notebook from a checked-out fair_forget repository containing src/fairforget/unlearning."
    )
sys.path.insert(0, str(repo / "src"))
from fairforget.unlearning import notebook_matrix, objectives, rmu  # noqa: E402, F401

print("Algorithm source:", repo / "src/fairforget/unlearning")
print("PyTorch:", torch.__version__)

METHOD = "rmu"
RUN_TRAINING = False

## Executable algorithm step

The following synthetic CPU example checks the implementation. Its tensors and learning rate are software examples, not study settings or measured model results.

In [ ]:
torch.manual_seed(42)
forget_activations = torch.nn.Parameter(torch.randn(2, 5, 8))
retain_activations = torch.nn.Parameter(torch.randn(2, 5, 8))
reference_retain = retain_activations.detach().clone()
mask = torch.tensor([[False, True, True, True], [False, True, True, False]])
control = rmu.fixed_control_vector(8, coefficient=20.0, seed=42)
optimizer = torch.optim.AdamW([forget_activations, retain_activations], lr=0.01)
optimizer.zero_grad(set_to_none=True)
loss = rmu.rmu_loss(
    forget_activations,
    retain_activations,
    reference_retain,
    control,
    mask,
    mask,
    retain_weight=1.0,
)
loss.total.backward()
assert torch.all(forget_activations.grad[:, 0] == 0)
assert torch.all(forget_activations.grad[:, 1:][~mask] == 0)
assert reference_retain.grad is None
optimizer.step()
print("Fixed-target RMU CPU optimizer step completed.")

## Approved full-model training

Every notebook covers Qwen3 8B, Llama 3.1 8B Instruct, Gemma 3 12B IT, Lapa 12B Instruct, Mamay 12B v2 and Mamay 27B v2. Set `FAIRFORGET_DATA_ROOT` to the approved data directory for this protocol, optionally set `FAIRFORGET_MODELS_ROOT` to the staged checkpoint root, and set `FAIRFORGET_OUTPUT` to a new method output directory. All six inputs pass validation before training starts. Original source hashes, model revisions, masks and development splits remain bound. Set `RUN_TRAINING=True` to run six independent trainers in sequence. Mamay 27B RMU needs two allocated GPUs; the other runs need one. Logs, receipts and checkpoints are saved separately for every model.

This notebook uses the **WarBias** protocol. See the [setup guide](SETUP.md) for installation, checkpoint downloads and data paths. The original NPO/BiasUnlearn primary data must be mounted externally; the reviewed WarBias data are bound by the published recipes. This model expansion changes no gradient source or objective setting.

In [ ]:
# Every method uses all six pinned models from the maintained registry.
# Optional roots rebase paths; original data hashes and model identities stay fixed.
DATA_ROOT = os.environ.get("FAIRFORGET_DATA_ROOT")
MODELS_ROOT = os.environ.get("FAIRFORGET_MODELS_ROOT")
settings_path = repo / "configs/unlearning/local.json"
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
output_value = os.environ.get("FAIRFORGET_OUTPUT") or str(
    Path(settings.get("output_root", repo / "outputs/unlearning")) / METHOD
)
OUTPUT = Path(output_value).expanduser().resolve()
PLAN = notebook_matrix.method_plan(repo, METHOD, data_root=DATA_ROOT, models_root=MODELS_ROOT)
assert len(PLAN) == 6, "This notebook must cover all six study models."
for item in PLAN:
    recipe = item["recipe"]
    print(
        item["model_alias"],
        recipe["model_id"],
        recipe["model_revision"],
        "GPUs:",
        item["required_gpus"],
    )
print("Models:", len(PLAN), "| Method:", METHOD)
print("Protocol:", PLAN[0]["recipe"]["notebook_matrix"]["protocol"])

In [ ]:
if RUN_TRAINING:
    if OUTPUT is None:
        raise ValueError("Set FAIRFORGET_OUTPUT to a new directory for all six model runs.")
    from importlib.metadata import version

    required_versions = {
        "torch": "2.9.1",
        "transformers": "4.57.6",
        "peft": "0.17.1",
        "accelerate": "1.15.0",
    }
    for package, expected in required_versions.items():
        if version(package).split("+")[0] != expected:
            raise RuntimeError(
                f"Use requirements-unlearning.txt: {package}=={expected} is required for full training."
            )
    if not torch.cuda.is_available() or torch.cuda.device_count() < max(
        item["required_gpus"] for item in PLAN
    ):
        raise RuntimeError(
            "Allocate the GPUs listed above; RMU for Mamay 27B needs two visible GPUs."
        )
    # Preflight validates ALL models, data hashes and reviewed roles before any training.
    recipe_paths = notebook_matrix.prepare_recipes(PLAN, OUTPUT)
    environment = dict(os.environ)
    environment["PYTHONPATH"] = str(repo / "src") + os.pathsep + environment.get("PYTHONPATH", "")
    environment.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    module = "fairforget.unlearning.rmu" if METHOD == "rmu" else "fairforget.unlearning.trainer"
    execution = {"method": METHOD, "models": {}}
    execution_path = OUTPUT / "matrix_execution.json"
    for item, recipe_path in zip(PLAN, recipe_paths, strict=True):
        alias = item["model_alias"]
        command = [
            sys.executable,
            "-m",
            module,
            "--config",
            str(recipe_path),
            "--output",
            str(OUTPUT / alias),
        ]
        record = {"status": "running", "recipe": str(recipe_path), "command": command}
        execution["models"][alias] = record
        execution_path.write_text(json.dumps(execution, indent=2) + "\n")
        print("Training", alias, flush=True)
        # Separate processes release GPU memory between complete model runs.
        with (OUTPUT / (alias + ".log")).open("w") as log:
            completed = subprocess.run(
                command,
                cwd=repo,
                env=environment,
                stdout=log,
                stderr=subprocess.STDOUT,
                check=False,
            )
        record.update(
            status="completed" if completed.returncode == 0 else "failed",
            returncode=completed.returncode,
        )
        execution_path.write_text(json.dumps(execution, indent=2) + "\n")
        completed.check_returncode()
    assert len(execution["models"]) == len(PLAN) == 6
    print("All six model runs completed:", execution_path)
else:
    print("All six models are configured. Full training starts only when RUN_TRAINING=True.")

## Implementation

- [Loss implementations](../src/fairforget/unlearning/objectives.py)
- [Full NPO, WGA, task-vector and BiasUnlearn trainer](../src/fairforget/unlearning/trainer.py)
- [Full fixed-target RMU trainer](../src/fairforget/unlearning/rmu.py)

The trainer writes the checkpoint, numeric history, actual valid-token exposure and provenance to the new output directory. Evaluation and reporting are separate workflows; use the [notebook index](README.md) to find saved reports.